# Chapter 3 — LayerNorm (`layernorm_forward` / `layernorm_backward`)

> Course: **llm.c — Zero to Hero**, Chapter 3 of ~20.
> Builds on Chapter 2 (forward+backward pattern, gradient buffers).

In Chapter 2 the layer was almost trivial: a gather plus an add. This chapter is the first time we have **real math** — per-row mean and variance, a normalize-scale-shift fused into one pass, and a backward pass with a derivation that catches almost everyone the first time they see it.

LayerNorm appears **twice in every Transformer block**: once before attention and once before the feed-forward (in pre-norm GPT-2). It's the workhorse that keeps activations from drifting and gradients from exploding. Understanding its backward properly is also a great exercise in chain rule with intermediate variables.

### Learning objectives

By the end of this chapter you will:

- State and code the LayerNorm forward in C from memory.
- Explain *why* `llm.c` caches `mean` and `rstd` from the forward — and what would go wrong without it.
- Derive the LayerNorm backward formula from first principles, then read the C version line-by-line.
- Understand the **two-pass** structure of the backward: why we *must* compute `dnorm_mean` and `dnorm_norm_mean` before we can write any `dinp[i]`.
- Distinguish which gradients are **per-position writes** (`dinp`) from which are **scatter-add accumulators** (`dweight`, `dbias`).


## 1. The Concept — What LayerNorm Actually Computes

Given a row of activations $x \in \mathbb{R}^C$ at one position $(b, t)$, LayerNorm computes:

$$\mu = \frac{1}{C}\sum_{i=1}^{C} x_i \qquad v = \frac{1}{C}\sum_{i=1}^{C} (x_i - \mu)^2$$

$$\hat x_i = \frac{x_i - \mu}{\sqrt{v + \varepsilon}} \qquad \text{out}_i = \gamma_i \hat x_i + \beta_i$$

with $\varepsilon = 10^{-5}$ for numerical stability. $\gamma$ (`weight`) and $\beta$ (`bias`) are learnable per-channel vectors of length $C$, **shared across all `(b,t)` positions**.

Three things that often confuse PyTorch users:

1. **Per-row, not per-batch.** Mean and variance are computed over the channel dimension `C`, separately for every `(b, t)`. There is no "running mean" like in BatchNorm — LayerNorm has no train/eval distinction.
2. **Variance is biased** (divides by `C`, not `C-1`). This matches PyTorch's `nn.LayerNorm` and makes the backward derivation cleaner.
3. **`weight` and `bias` are shape `(C,)`**, not `(B, T, C)`. They are broadcast over batch and time.

The "reciprocal standard deviation" $r = 1/\sqrt{v + \varepsilon}$ shows up so often we cache it under the name `rstd`. Storing $r$ is *strictly more useful* than storing $v$ — every backward formula uses $r$ directly, never $v$.


## 2. PyTorch Baseline

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
B, T, C = 2, 4, 8

x  = torch.randn(B, T, C)
ln = nn.LayerNorm(C)            # weight=ones(C), bias=zeros(C) by default
nn.init.normal_(ln.weight); nn.init.normal_(ln.bias)  # randomize so we can really test

out = ln(x)
print("input  shape:", x.shape)
print("output shape:", out.shape)
print("per-row mean of out (should be ~bias):", out.mean(dim=-1)[0, 0].item(), "vs bias.mean", ln.bias.mean().item())

# Verify by hand on one row
row = x[0, 0]
mu  = row.mean()
v   = ((row - mu)**2).mean()
nrm = (row - mu) / torch.sqrt(v + 1e-5)
manual = nrm * ln.weight + ln.bias
print("max diff between nn.LayerNorm and hand math:", (out[0,0] - manual).abs().max().item())


Note that `nn.LayerNorm(C)` defaults to `eps=1e-5` and `elementwise_affine=True` (i.e., it has `weight` and `bias`). PyTorch's variance is biased (divided by `C`), matching `llm.c`.


## 3. The C Forward Pass

Here is `layernorm_forward` from [`train_gpt2.c`](train_gpt2.c) (lines 78–118), copied verbatim:

```c
void layernorm_forward(float* out, float* mean, float* rstd,
                       float* inp, float* weight, float* bias,
                       int B, int T, int C) {
    float eps = 1e-5f;
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            // seek to the input position inp[b,t,:]
            float* x = inp + b * T * C + t * C;
            // calculate the mean
            float m = 0.0f;
            for (int i = 0; i < C; i++) {
                m += x[i];
            }
            m = m/C;
            // calculate the variance (without any bias correction)
            float v = 0.0f;
            for (int i = 0; i < C; i++) {
                float xshift = x[i] - m;
                v += xshift * xshift;
            }
            v = v/C;
            // calculate the rstd (reciprocal standard deviation)
            float s = 1.0f / sqrtf(v + eps);
            // seek to the output position in out[b,t,:]
            float* out_bt = out + b * T * C + t * C;
            for (int i = 0; i < C; i++) {
                float n = (s * (x[i] - m)); // normalize
                float o = n * weight[i] + bias[i]; // scale and shift
                out_bt[i] = o; // write
            }
            // cache the mean and rstd for the backward pass later
            mean[b * T + t] = m;
            rstd[b * T + t] = s;
        }
    }
}
```

Walk through it:

| Block | What it does |
|---|---|
| `float* x = inp + b*T*C + t*C;` | Row pointer to `inp[b, t, :]` (Chapter 1 idiom). |
| 1st inner loop | Compute the mean `m = (1/C) Σ x[i]`. |
| 2nd inner loop | Compute the variance `v = (1/C) Σ (x[i] - m)²`. |
| `s = 1.0f / sqrtf(v + eps);` | The *reciprocal* standard deviation `rstd`. We store `1/σ`, not `σ`, because that is what gets multiplied later. |
| 3rd inner loop | Fused normalize-scale-shift: `out[i] = (x[i]-m)*s * γ[i] + β[i]`. |
| `mean[b*T+t] = m; rstd[b*T+t] = s;` | **Cache `m` and `s` for backward.** This is the single most important line for backward efficiency. |

### Three things to notice

1. **Three passes over `x`**, but reuse of the `out_bt` row pointer. The compiler & CPU caches make this fast: `x` stays in L1 across the three loops because `C ≤ 768` for GPT-2 small (3 KB).
2. `mean` and `rstd` are tensors of shape `(B, T)` — one scalar each per row. They live as **separate buffers** alongside `out`. In `llm.c` they're slots in the giant `ActivationTensors` struct (we'll see this in Chapter 8).
3. `weight` and `bias` are shape `(C,)` and are accessed as `weight[i]` / `bias[i]` directly — no `b` or `t` indexing. They are **shared parameters**.


## 4. Why Cache `mean` and `rstd`?

Two reasons, one obvious and one subtle.

**Obvious:** the backward needs them, and recomputing them costs another two passes over `x` *per row*. Caching turns backward into a single pass plus reading two small numbers — much cheaper than recomputing.

**Subtle:** caching `rstd = 1/sqrt(v+eps)` rather than `v` keeps **every backward expression a multiplication** instead of a division. Look at the backward we're about to derive: every place where the math says "divide by σ", the code multiplies by `rstd`. CPUs and GPUs are *much* faster at multiplying than dividing, and divisions also cause numerical issues near zero. Caching the reciprocal is the difference between "fast and stable" and "slow and surprising".

This caching pattern reappears throughout `llm.c`: any time the forward computes a quantity the backward will need, it's stashed. Memory used = activations memory. Time saved = the forward's worth of recomputation per backward.


## 5. Translation Bridge

| PyTorch | C in `llm.c` |
|---|---|
| `nn.LayerNorm(C)` (module with `weight`, `bias`) | Three pointers: `weight`, `bias`, plus stat buffers `mean`, `rstd` you allocate yourself |
| `eps=1e-5` default | hardcoded `float eps = 1e-5f;` |
| Variance via `var(unbiased=False)` | hand loop with `/C` (not `/C-1`) |
| Output reshape/broadcast handled internally | The C code visits one `(b,t)` row at a time, no broadcasting machinery |
| Stats stashed automatically by autograd | You explicitly write `mean[b*T+t]` and `rstd[b*T+t]` |
| `out = ln(x)` | one call to `layernorm_forward(...)` with 8 pointer/int args |

Mental model: **PyTorch's `nn.LayerNorm` is a module that hides 4 buffers (`weight`, `bias`, intermediate `mean` and `rstd`). In C those are 4 raw pointers and you wire them up by hand.**


## 6. Compile, Run, and Cross-Check Forward

In [ ]:
!mkdir -p course/ch03_build


In [ ]:
%%writefile course/ch03_build/layernorm_forward.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

void layernorm_forward(float* out, float* mean, float* rstd,
                       float* inp, float* weight, float* bias,
                       int B, int T, int C) {
    float eps = 1e-5f;
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* x = inp + b*T*C + t*C;
            float m = 0.0f;
            for (int i = 0; i < C; i++) m += x[i];
            m = m/C;
            float v = 0.0f;
            for (int i = 0; i < C; i++) { float d = x[i]-m; v += d*d; }
            v = v/C;
            float s = 1.0f / sqrtf(v + eps);
            float* out_bt = out + b*T*C + t*C;
            for (int i = 0; i < C; i++) {
                float n = s * (x[i] - m);
                out_bt[i] = n * weight[i] + bias[i];
            }
            mean[b*T + t] = m;
            rstd[b*T + t] = s;
        }
    }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f) { perror(p); exit(1); }
    void* b = malloc(n); size_t r = fread(b, 1, n, f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 4) { fprintf(stderr, "usage: B T C\n"); return 1; }
    int B = atoi(argv[1]), T = atoi(argv[2]), C = atoi(argv[3]);

    float* inp    = (float*) rd("course/ch03_build/inp.bin",    (size_t)B*T*C*sizeof(float));
    float* weight = (float*) rd("course/ch03_build/weight.bin", (size_t)C*sizeof(float));
    float* bias   = (float*) rd("course/ch03_build/bias.bin",   (size_t)C*sizeof(float));
    float* out    = (float*) malloc((size_t)B*T*C*sizeof(float));
    float* mean   = (float*) malloc((size_t)B*T*sizeof(float));
    float* rstd   = (float*) malloc((size_t)B*T*sizeof(float));

    layernorm_forward(out, mean, rstd, inp, weight, bias, B, T, C);

    FILE* f;
    f = fopen("course/ch03_build/out.bin",  "wb"); fwrite(out,  4, (size_t)B*T*C, f); fclose(f);
    f = fopen("course/ch03_build/mean.bin", "wb"); fwrite(mean, 4, (size_t)B*T,   f); fclose(f);
    f = fopen("course/ch03_build/rstd.bin", "wb"); fwrite(rstd, 4, (size_t)B*T,   f); fclose(f);

    free(inp); free(weight); free(bias); free(out); free(mean); free(rstd);
    return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch03_build/layernorm_forward course/ch03_build/layernorm_forward.c -lm


In [ ]:
# Generate inputs in Python, run our C, compare to nn.LayerNorm
import numpy as np, torch, torch.nn as nn, subprocess

torch.manual_seed(0)
B, T, C = 2, 4, 8

x = torch.randn(B, T, C)
ln = nn.LayerNorm(C)
nn.init.normal_(ln.weight); nn.init.normal_(ln.bias)

x.numpy().astype(np.float32).tofile("course/ch03_build/inp.bin")
ln.weight.detach().numpy().astype(np.float32).tofile("course/ch03_build/weight.bin")
ln.bias.detach().numpy().astype(np.float32).tofile("course/ch03_build/bias.bin")

subprocess.run(["./course/ch03_build/layernorm_forward", str(B), str(T), str(C)], check=True)

out_c  = np.fromfile("course/ch03_build/out.bin",  dtype=np.float32).reshape(B, T, C)
mean_c = np.fromfile("course/ch03_build/mean.bin", dtype=np.float32).reshape(B, T)
rstd_c = np.fromfile("course/ch03_build/rstd.bin", dtype=np.float32).reshape(B, T)

with torch.no_grad():
    out_pt  = ln(x).numpy()
    mean_pt = x.mean(dim=-1).numpy()
    var_pt  = x.var(dim=-1, unbiased=False).numpy()
    rstd_pt = 1.0 / np.sqrt(var_pt + 1e-5)

print(f"out  max diff: {np.max(np.abs(out_c  - out_pt )):.2e}")
print(f"mean max diff: {np.max(np.abs(mean_c - mean_pt)):.2e}")
print(f"rstd max diff: {np.max(np.abs(rstd_c - rstd_pt)):.2e}")


All three buffers should match PyTorch to roughly `1e-6` (float32 noise). **You just ran LayerNorm in C, including the cached `mean` and `rstd`.**


## 7. The Backward — Math First (Skip if You Just Want the Code)

Setup. For one row, write $r = $`rstd`, $\mu = $`mean`. Then:

$$\hat x_i = (x_i - \mu) r \qquad \text{out}_i = \gamma_i \hat x_i + \beta_i$$

We're given $\partial L / \partial \text{out}_i$ (call this `dout[i]`) and want $\partial L / \partial x_i$, $\partial L / \partial \gamma_i$, $\partial L / \partial \beta_i$.

### The easy two

By direct differentiation of $\text{out}_i = \gamma_i \hat x_i + \beta_i$:

$$\frac{\partial L}{\partial \beta_i} = \text{dout}_i \qquad \frac{\partial L}{\partial \gamma_i} = \hat x_i \cdot \text{dout}_i$$

These are summed over **all** `(b, t)` positions, since `weight` and `bias` are shared. That's a scatter-add pattern again, just like `dwte` in Chapter 2 — but here every `(b,t)` contributes to every channel of `dweight`/`dbias`, not just one row.

### The hard one — `dinp`

Define an intermediate gradient $d\hat x_j = \gamma_j \cdot \text{dout}_j$ (the gradient flowing *into* the normalized values). Now:

$$\frac{\partial L}{\partial x_i} = \sum_j d\hat x_j \cdot \frac{\partial \hat x_j}{\partial x_i}$$

Compute $\partial \hat x_j / \partial x_i$ by differentiating $\hat x_j = (x_j - \mu) r$ with respect to $x_i$. Both $\mu$ and $r$ depend on every $x_i$:

- $\partial \mu / \partial x_i = 1/C$
- $\partial v / \partial x_i = (2/C)(x_i - \mu)$  (after using $\sum (x-\mu) = 0$)
- $\partial r / \partial x_i = -\frac{1}{2}(v+\varepsilon)^{-3/2} \cdot \partial v / \partial x_i = -\frac{r}{C} \hat x_i$

Plug in:

$$\frac{\partial \hat x_j}{\partial x_i} = (\delta_{ij} - 1/C)\, r - \frac{r}{C} \hat x_i \hat x_j$$

Sum over $j$:

$$\frac{\partial L}{\partial x_i} = r\Big[d\hat x_i - \tfrac{1}{C}\sum_j d\hat x_j - \hat x_i \cdot \tfrac{1}{C}\sum_j d\hat x_j \hat x_j\Big]$$

Define two **per-row reductions** (averages):

$$\overline{d\hat x} = \frac{1}{C}\sum_j d\hat x_j \qquad \overline{d\hat x \cdot \hat x} = \frac{1}{C}\sum_j d\hat x_j \hat x_j$$

Then:

$$\boxed{\frac{\partial L}{\partial x_i} = r\Big(d\hat x_i - \overline{d\hat x} - \hat x_i \cdot \overline{d\hat x \cdot \hat x}\Big)}$$

This is the formula the C code implements. The two averages are **per-row scalars** that cannot be computed elementwise — you must visit *all* `j` to know either of them. **That is why the backward needs two passes**: one to compute the two averages, one to write each `dinp[i]`.


## 8. The C Backward — Line by Line

From [`train_gpt2.c`](train_gpt2.c) lines 120–161:

```c
void layernorm_backward(float* dinp, float* dweight, float* dbias,
                        float* dout, float* inp, float* weight, float* mean, float* rstd,
                        int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b * T * C + t * C;
            float* inp_bt  = inp  + b * T * C + t * C;
            float* dinp_bt = dinp + b * T * C + t * C;
            float mean_bt  = mean[b * T + t];
            float rstd_bt  = rstd[b * T + t];

            // PASS 1: two reductions over the C dimension
            float dnorm_mean = 0.0f;
            float dnorm_norm_mean = 0.0f;
            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;   // = x_hat_i
                float dnorm_i  = weight[i] * dout_bt[i];            // = d_x_hat_i
                dnorm_mean      += dnorm_i;
                dnorm_norm_mean += dnorm_i * norm_bti;
            }
            dnorm_mean       = dnorm_mean / C;        // mean of dnorm_i
            dnorm_norm_mean  = dnorm_norm_mean / C;   // mean of dnorm_i * norm_i

            // PASS 2: write dinp[i], accumulate dweight, dbias
            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                float dnorm_i  = weight[i] * dout_bt[i];

                dbias[i]   += dout_bt[i];                  // dL/d beta_i
                dweight[i] += norm_bti * dout_bt[i];        // dL/d gamma_i

                float dval = 0.0f;
                dval += dnorm_i;                            // term 1
                dval -= dnorm_mean;                         // term 2
                dval -= norm_bti * dnorm_norm_mean;         // term 3
                dval *= rstd_bt;                            // multiply by r at the end
                dinp_bt[i] += dval;                         // dL/d x_i
            }
        }
    }
}
```

Map this to the formula:

| Code variable        | Math symbol |
|----------------------|-------------|
| `norm_bti`           | $\hat x_i$ (recomputed since we didn't cache it — only $\mu, r$) |
| `dnorm_i`            | $d\hat x_i = \gamma_i \cdot \text{dout}_i$ |
| `dnorm_mean`         | $\overline{d\hat x}$ |
| `dnorm_norm_mean`    | $\overline{d\hat x \cdot \hat x}$ |
| `dval`               | $\partial L / \partial x_i$ |

### Three things worth noticing

1. **`norm_bti` is recomputed twice** — once in pass 1, once in pass 2. We could cache `norm` (a `(B,T,C)` tensor) but that *doubles* activation memory. The C version pays a few FLOPs to save bytes. (The CUDA version may make a different trade-off.)
2. **`dweight` and `dbias` use `+=`** — same scatter-add story as `dwte` in Chapter 2. Every `(b,t)` contributes. They must be zeroed before this function is called.
3. **`dinp_bt[i] += dval` not `=`** — because in `llm.c` the same `dinp` buffer is used across the residual stream. The "previous" backward step has already written into it (from the residual connection), and this layer's gradient adds on top.


## 9. Compile, Run, Verify Against Autograd

In [ ]:
%%writefile course/ch03_build/layernorm_backward.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

void layernorm_backward(float* dinp, float* dweight, float* dbias,
                        float* dout, float* inp, float* weight, float* mean, float* rstd,
                        int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b*T*C + t*C;
            float* inp_bt  = inp  + b*T*C + t*C;
            float* dinp_bt = dinp + b*T*C + t*C;
            float mean_bt  = mean[b*T + t];
            float rstd_bt  = rstd[b*T + t];

            float dnorm_mean = 0.0f, dnorm_norm_mean = 0.0f;
            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                float dnorm_i  = weight[i] * dout_bt[i];
                dnorm_mean      += dnorm_i;
                dnorm_norm_mean += dnorm_i * norm_bti;
            }
            dnorm_mean      /= C;
            dnorm_norm_mean /= C;

            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                float dnorm_i  = weight[i] * dout_bt[i];
                dbias[i]   += dout_bt[i];
                dweight[i] += norm_bti * dout_bt[i];
                float dval = dnorm_i - dnorm_mean - norm_bti * dnorm_norm_mean;
                dval *= rstd_bt;
                dinp_bt[i] += dval;
            }
        }
    }
}

static void* rd(const char* p, size_t n) {
    FILE* f = fopen(p, "rb"); if (!f) { perror(p); exit(1); }
    void* b = malloc(n); size_t r = fread(b,1,n,f); (void)r; fclose(f); return b;
}

int main(int argc, char** argv) {
    if (argc != 4) return 1;
    int B = atoi(argv[1]), T = atoi(argv[2]), C = atoi(argv[3]);

    float* inp    = (float*) rd("course/ch03_build/inp.bin",    (size_t)B*T*C*sizeof(float));
    float* weight = (float*) rd("course/ch03_build/weight.bin", (size_t)C*sizeof(float));
    float* mean   = (float*) rd("course/ch03_build/mean.bin",   (size_t)B*T*sizeof(float));
    float* rstd   = (float*) rd("course/ch03_build/rstd.bin",   (size_t)B*T*sizeof(float));
    float* dout   = (float*) rd("course/ch03_build/dout.bin",   (size_t)B*T*C*sizeof(float));

    float* dinp    = (float*) calloc((size_t)B*T*C, sizeof(float));
    float* dweight = (float*) calloc((size_t)C,     sizeof(float));
    float* dbias   = (float*) calloc((size_t)C,     sizeof(float));

    layernorm_backward(dinp, dweight, dbias, dout, inp, weight, mean, rstd, B, T, C);

    FILE* f;
    f = fopen("course/ch03_build/dinp.bin",    "wb"); fwrite(dinp,    4, (size_t)B*T*C, f); fclose(f);
    f = fopen("course/ch03_build/dweight.bin", "wb"); fwrite(dweight, 4, (size_t)C,     f); fclose(f);
    f = fopen("course/ch03_build/dbias.bin",   "wb"); fwrite(dbias,   4, (size_t)C,     f); fclose(f);

    free(inp); free(weight); free(mean); free(rstd); free(dout);
    free(dinp); free(dweight); free(dbias);
    return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch03_build/layernorm_backward course/ch03_build/layernorm_backward.c -lm


In [ ]:
# Run autograd, save inputs+stats, run our C backward, compare
import numpy as np, torch, torch.nn as nn, subprocess

torch.manual_seed(0)
B, T, C = 2, 4, 8

x  = torch.randn(B, T, C, requires_grad=True)
ln = nn.LayerNorm(C)
nn.init.normal_(ln.weight); nn.init.normal_(ln.bias)
out  = ln(x)
dout = torch.randn_like(out)
out.backward(dout)

# Save everything our C binary needs
x.detach().numpy().astype(np.float32).tofile("course/ch03_build/inp.bin")
ln.weight.detach().numpy().astype(np.float32).tofile("course/ch03_build/weight.bin")
ln.bias.detach().numpy().astype(np.float32).tofile("course/ch03_build/bias.bin")
dout.numpy().astype(np.float32).tofile("course/ch03_build/dout.bin")
# Re-run our forward to fill mean/rstd cache files
subprocess.run(["./course/ch03_build/layernorm_forward", str(B), str(T), str(C)], check=True)

# Now run the backward
subprocess.run(["./course/ch03_build/layernorm_backward", str(B), str(T), str(C)], check=True)

dinp_c    = np.fromfile("course/ch03_build/dinp.bin",    dtype=np.float32).reshape(B, T, C)
dweight_c = np.fromfile("course/ch03_build/dweight.bin", dtype=np.float32).reshape(C)
dbias_c   = np.fromfile("course/ch03_build/dbias.bin",   dtype=np.float32).reshape(C)

print(f"dinp    max diff: {np.max(np.abs(dinp_c    - x.grad.numpy()        )):.2e}")
print(f"dweight max diff: {np.max(np.abs(dweight_c - ln.weight.grad.numpy())):.2e}")
print(f"dbias   max diff: {np.max(np.abs(dbias_c   - ln.bias.grad.numpy()  )):.2e}")


All three should match autograd to ~`1e-6` (float32 noise). The hardest gradient — `dinp` — uses a formula a lot of people get wrong on the whiteboard, but it's only 4 lines of C.


## 10. Toy Example — The Two Reductions Made Concrete

The two scalars `dnorm_mean` and `dnorm_norm_mean` are the "subtle" part of the backward. Let's run a single-row example with `C=4` and print them so you can see the math line up.


In [ ]:
%%writefile course/ch03_build/toy_reduce.c
#include <stdio.h>
#include <math.h>

int main(void) {
    const int C = 4;
    float x[4]      = {1.0f, 2.0f, 3.0f, 4.0f};      // input row
    float weight[4] = {0.5f, 1.0f, 1.5f, 2.0f};      // per-channel scale
    float dout[4]   = {0.1f, 0.2f, 0.3f, 0.4f};      // upstream gradient

    // Forward
    float m = 0.0f; for (int i=0;i<C;i++) m += x[i]; m /= C;
    float v = 0.0f; for (int i=0;i<C;i++){float d=x[i]-m; v += d*d;} v /= C;
    float r = 1.0f / sqrtf(v + 1e-5f);
    printf("mean=%.4f, var=%.4f, rstd=%.4f\n", m, v, r);

    // Backward pass 1: the two averages
    float dnorm_mean = 0.0f, dnorm_norm_mean = 0.0f;
    for (int i = 0; i < C; i++) {
        float norm_i  = (x[i] - m) * r;
        float dnorm_i = weight[i] * dout[i];
        printf("  i=%d  x_hat=%+.4f  d_x_hat=%+.4f  d_x_hat*x_hat=%+.4f\n",
               i, norm_i, dnorm_i, dnorm_i * norm_i);
        dnorm_mean      += dnorm_i;
        dnorm_norm_mean += dnorm_i * norm_i;
    }
    dnorm_mean /= C; dnorm_norm_mean /= C;
    printf("dnorm_mean         = %+.4f   (1/C * sum d_x_hat)\n", dnorm_mean);
    printf("dnorm_norm_mean    = %+.4f   (1/C * sum d_x_hat * x_hat)\n", dnorm_norm_mean);

    // Backward pass 2: the dinp formula
    printf("dinp_i = rstd * (d_x_hat_i - dnorm_mean - x_hat_i * dnorm_norm_mean):\n");
    for (int i = 0; i < C; i++) {
        float norm_i  = (x[i] - m) * r;
        float dnorm_i = weight[i] * dout[i];
        float dval = (dnorm_i - dnorm_mean - norm_i * dnorm_norm_mean) * r;
        printf("  dinp[%d] = %+.6f\n", i, dval);
    }
    return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch03_build/toy_reduce course/ch03_build/toy_reduce.c -lm && ./course/ch03_build/toy_reduce


In [ ]:
# Verify the toy by hand using PyTorch autograd on the same numbers
import torch, torch.nn as nn
x = torch.tensor([1.,2.,3.,4.], requires_grad=True)
ln = nn.LayerNorm(4, elementwise_affine=True)
with torch.no_grad():
    ln.weight.copy_(torch.tensor([0.5, 1.0, 1.5, 2.0]))
    ln.bias.zero_()
out = ln(x)
out.backward(torch.tensor([0.1, 0.2, 0.3, 0.4]))
print("autograd dinp:", x.grad.tolist())


The C printout's last block (`dinp[0..3]`) should match the autograd values to last-bit float32. You can now point at any line of `layernorm_backward` and say what it does.


## 11. TODO Exercise 1 — Write `layernorm_forward`

Boilerplate provided. Fill in the four numbered TODOs.


In [ ]:
%%writefile course/ch03_build/exercise1.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

void layernorm_forward(float* out, float* mean, float* rstd,
                       float* inp, float* weight, float* bias,
                       int B, int T, int C) {
    float eps = 1e-5f;
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* x = inp + b*T*C + t*C;

            // TODO 1: compute the mean m of x[0..C-1]
            float m = 0.0f;
            for (int i = 0; i <C; i++) {m += x[i];}
            m = m / C;
            // <-- your code here

            // TODO 2: compute the (biased) variance v
            float v = 0.0f;
            for (int i = 0; i <C; i++) {float d = x[i] - m; v += d * d;}
            v = v / C;
            // <-- your code here

            // TODO 3: compute the rstd s = 1 / sqrt(v + eps)
            float s = 1 / sqrt(v + eps);  // <-- replace

            // TODO 4: write out_bt[i] = s*(x[i]-m) * weight[i] + bias[i]
            float* out_bt = out + b*T*C + t*C;
            for (int i = 0; i < C; i++) {
                out_bt[i] = s * (x[i] - m) * weight[i] + bias[i];  // <-- replace
            }

            // (provided) cache mean and rstd for backward
            mean[b*T + t] = m;
            rstd[b*T + t] = s;
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]);
    float* inp    = (float*) rd("course/ch03_build/inp.bin",    (size_t)B*T*C*sizeof(float));
    float* weight = (float*) rd("course/ch03_build/weight.bin", (size_t)C*sizeof(float));
    float* bias   = (float*) rd("course/ch03_build/bias.bin",   (size_t)C*sizeof(float));
    float* out    = (float*) malloc((size_t)B*T*C*sizeof(float));
    float* mean   = (float*) malloc((size_t)B*T*sizeof(float));
    float* rstd   = (float*) malloc((size_t)B*T*sizeof(float));
    layernorm_forward(out, mean, rstd, inp, weight, bias, B, T, C);
    FILE* f = fopen("course/ch03_build/out_ex1.bin","wb"); fwrite(out, 4, (size_t)B*T*C, f); fclose(f);
    free(inp); free(weight); free(bias); free(out); free(mean); free(rstd); return 0;
}


In [ ]:
# Auto-grader for Exercise 1
import numpy as np, torch, torch.nn as nn, subprocess
torch.manual_seed(0); B, T, C = 2, 4, 8
x = torch.randn(B, T, C); ln = nn.LayerNorm(C)
nn.init.normal_(ln.weight); nn.init.normal_(ln.bias)
x.numpy().astype(np.float32).tofile("course/ch03_build/inp.bin")
ln.weight.detach().numpy().astype(np.float32).tofile("course/ch03_build/weight.bin")
ln.bias.detach().numpy().astype(np.float32).tofile("course/ch03_build/bias.bin")
subprocess.run(["gcc","-O2","-Wall","-o","course/ch03_build/exercise1","course/ch03_build/exercise1.c","-lm"], check=True)
subprocess.run(["./course/ch03_build/exercise1", str(B), str(T), str(C)], check=True)
out_ex = np.fromfile("course/ch03_build/out_ex1.bin", dtype=np.float32).reshape(B, T, C)
out_pt = ln(x).detach().numpy()
err = np.max(np.abs(out_ex - out_pt))
print(f"max abs diff: {err:.2e}")
print("PASS" if err < 1e-5 else "FAIL — check mean / variance / rstd / scale-shift")


### Solution to Exercise 1

In [ ]:
%%writefile course/ch03_build/exercise1_sol.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

void layernorm_forward(float* out, float* mean, float* rstd,
                       float* inp, float* weight, float* bias,
                       int B, int T, int C) {
    float eps = 1e-5f;
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* x = inp + b*T*C + t*C;
            float m = 0.0f;
            for (int i = 0; i < C; i++) m += x[i];
            m /= C;
            float v = 0.0f;
            for (int i = 0; i < C; i++) { float d = x[i] - m; v += d*d; }
            v /= C;
            float s = 1.0f / sqrtf(v + eps);
            float* out_bt = out + b*T*C + t*C;
            for (int i = 0; i < C; i++) {
                out_bt[i] = s * (x[i] - m) * weight[i] + bias[i];
            }
            mean[b*T + t] = m;
            rstd[b*T + t] = s;
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]);
    float* inp    = (float*) rd("course/ch03_build/inp.bin",    (size_t)B*T*C*sizeof(float));
    float* weight = (float*) rd("course/ch03_build/weight.bin", (size_t)C*sizeof(float));
    float* bias   = (float*) rd("course/ch03_build/bias.bin",   (size_t)C*sizeof(float));
    float* out    = (float*) malloc((size_t)B*T*C*sizeof(float));
    float* mean   = (float*) malloc((size_t)B*T*sizeof(float));
    float* rstd   = (float*) malloc((size_t)B*T*sizeof(float));
    layernorm_forward(out, mean, rstd, inp, weight, bias, B, T, C);
    FILE* f = fopen("course/ch03_build/out_ex1.bin","wb"); fwrite(out, 4, (size_t)B*T*C, f); fclose(f);
    free(inp); free(weight); free(bias); free(out); free(mean); free(rstd); return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch03_build/exercise1_sol course/ch03_build/exercise1_sol.c -lm && ./course/ch03_build/exercise1_sol 2 4 8 && echo "ran"


## 12. TODO Exercise 2 — Backward for `dweight` and `dbias` Only

The `dinp` formula is the trickiest of the three. Let's start with the easier two: **`dbias[i] += dout_bt[i]`** and **`dweight[i] += norm_bti * dout_bt[i]`**, both summed over all `(b, t)` positions.

Boilerplate computes `norm_bti` for you; you fill in the two `+=` lines.


In [ ]:
%%writefile course/ch03_build/exercise2.c
#include <stdio.h>
#include <stdlib.h>

void layernorm_backward_dwdb(float* dweight, float* dbias,
                             float* dout, float* inp,
                             float* mean, float* rstd,
                             int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b*T*C + t*C;
            float* inp_bt  = inp  + b*T*C + t*C;
            float mean_bt  = mean[b*T + t];
            float rstd_bt  = rstd[b*T + t];
            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                // TODO: accumulate into dbias[i]    (hint: just dout_bt[i])
                dbias[i] += dout_bt[i];
                // TODO: accumulate into dweight[i]  (hint: norm_bti * dout_bt[i])
                dweight[i] += norm_bti * dout_bt[i];
            }
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]);
    float* inp  = (float*) rd("course/ch03_build/inp.bin",  (size_t)B*T*C*sizeof(float));
    float* mean = (float*) rd("course/ch03_build/mean.bin", (size_t)B*T*sizeof(float));
    float* rstd = (float*) rd("course/ch03_build/rstd.bin", (size_t)B*T*sizeof(float));
    float* dout = (float*) rd("course/ch03_build/dout.bin", (size_t)B*T*C*sizeof(float));
    float* dweight = (float*) calloc((size_t)C, sizeof(float));
    float* dbias   = (float*) calloc((size_t)C, sizeof(float));
    layernorm_backward_dwdb(dweight, dbias, dout, inp, mean, rstd, B, T, C);
    FILE* f;
    f = fopen("course/ch03_build/dweight_ex2.bin","wb"); fwrite(dweight,4,(size_t)C,f); fclose(f);
    f = fopen("course/ch03_build/dbias_ex2.bin",  "wb"); fwrite(dbias,  4,(size_t)C,f); fclose(f);
    free(inp); free(mean); free(rstd); free(dout); free(dweight); free(dbias); return 0;
}


In [ ]:
# Auto-grader: compare against autograd's dweight & dbias
import numpy as np, torch, torch.nn as nn, subprocess
torch.manual_seed(0); B, T, C = 2, 4, 8
x  = torch.randn(B, T, C, requires_grad=True)
ln = nn.LayerNorm(C); nn.init.normal_(ln.weight); nn.init.normal_(ln.bias)
out = ln(x); dout = torch.randn_like(out); out.backward(dout)
x.detach().numpy().astype(np.float32).tofile("course/ch03_build/inp.bin")
ln.weight.detach().numpy().astype(np.float32).tofile("course/ch03_build/weight.bin")
ln.bias.detach().numpy().astype(np.float32).tofile("course/ch03_build/bias.bin")
dout.numpy().astype(np.float32).tofile("course/ch03_build/dout.bin")
subprocess.run(["./course/ch03_build/layernorm_forward", str(B), str(T), str(C)], check=True)  # refresh mean.bin, rstd.bin
subprocess.run(["gcc","-O2","-Wall","-o","course/ch03_build/exercise2","course/ch03_build/exercise2.c"], check=True)
subprocess.run(["./course/ch03_build/exercise2", str(B), str(T), str(C)], check=True)
dweight_ex = np.fromfile("course/ch03_build/dweight_ex2.bin", dtype=np.float32)
dbias_ex   = np.fromfile("course/ch03_build/dbias_ex2.bin",   dtype=np.float32)
e1 = np.max(np.abs(dweight_ex - ln.weight.grad.numpy()))
e2 = np.max(np.abs(dbias_ex   - ln.bias.grad.numpy()))
print(f"dweight diff: {e1:.2e}\ndbias   diff: {e2:.2e}")
print("PASS" if max(e1,e2) < 1e-5 else "FAIL — check the two += lines")


### Solution to Exercise 2

In [ ]:
%%writefile course/ch03_build/exercise2_sol.c
#include <stdio.h>
#include <stdlib.h>

void layernorm_backward_dwdb(float* dweight, float* dbias,
                             float* dout, float* inp,
                             float* mean, float* rstd,
                             int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b*T*C + t*C;
            float* inp_bt  = inp  + b*T*C + t*C;
            float mean_bt  = mean[b*T + t];
            float rstd_bt  = rstd[b*T + t];
            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                dbias[i]   += dout_bt[i];
                dweight[i] += norm_bti * dout_bt[i];
            }
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]);
    float* inp  = (float*) rd("course/ch03_build/inp.bin",  (size_t)B*T*C*sizeof(float));
    float* mean = (float*) rd("course/ch03_build/mean.bin", (size_t)B*T*sizeof(float));
    float* rstd = (float*) rd("course/ch03_build/rstd.bin", (size_t)B*T*sizeof(float));
    float* dout = (float*) rd("course/ch03_build/dout.bin", (size_t)B*T*C*sizeof(float));
    float* dweight = (float*) calloc((size_t)C, sizeof(float));
    float* dbias   = (float*) calloc((size_t)C, sizeof(float));
    layernorm_backward_dwdb(dweight, dbias, dout, inp, mean, rstd, B, T, C);
    FILE* f;
    f = fopen("course/ch03_build/dweight_ex2.bin","wb"); fwrite(dweight,4,(size_t)C,f); fclose(f);
    f = fopen("course/ch03_build/dbias_ex2.bin",  "wb"); fwrite(dbias,  4,(size_t)C,f); fclose(f);
    free(inp); free(mean); free(rstd); free(dout); free(dweight); free(dbias); return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch03_build/exercise2_sol course/ch03_build/exercise2_sol.c && ./course/ch03_build/exercise2_sol 2 4 8 && echo ran


## 13. (Stretch) TODO Exercise 3 — Full `dinp`

If you're feeling brave: write the full `dinp` formula. You need to:

1. Pass 1: compute `dnorm_mean` and `dnorm_norm_mean` (two scalars per row).
2. Pass 2: write `dinp_bt[i] += rstd_bt * (dnorm_i - dnorm_mean - norm_bti * dnorm_norm_mean)`.

Skip this if you'd rather move on to Chapter 4 — the formula is on display in section 8 and you can always come back. Solution provided below as usual.


In [ ]:
%%writefile course/ch03_build/exercise3.c
#include <stdio.h>
#include <stdlib.h>

void layernorm_backward_dinp(float* dinp,
                             float* dout, float* inp, float* weight,
                             float* mean, float* rstd,
                             int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b*T*C + t*C;
            float* inp_bt  = inp  + b*T*C + t*C;
            float* dinp_bt = dinp + b*T*C + t*C;
            float mean_bt  = mean[b*T + t];
            float rstd_bt  = rstd[b*T + t];

            // TODO pass 1: compute dnorm_mean and dnorm_norm_mean
            float dnorm_mean = 0.0f, dnorm_norm_mean = 0.0f;
            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                float dnorm_i  = weight[i] * dout_bt[i];
                dnorm_mean += dnorm_i;
                dnorm_norm_mean += dnorm_i * norm_bti;
                // <-- accumulate dnorm_mean and dnorm_norm_mean
                (void)norm_bti; (void)dnorm_i;
            }
            dnorm_mean      /= C;
            dnorm_norm_mean /= C;

            // TODO pass 2: write dinp_bt[i] += rstd * (dnorm_i - dnorm_mean - norm_bti*dnorm_norm_mean)
            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                float dnorm_i  = weight[i] * dout_bt[i];
                float dval = 0.0f;
                dval += dnorm_i;
                dval -= dnorm_mean;
                dval -= norm_bti * dnorm_norm_mean;
                dval *= rstd_bt;
                
                // <-- compute dval
                dinp_bt[i] += dval;
            }
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]);
    float* inp    = (float*) rd("course/ch03_build/inp.bin",    (size_t)B*T*C*sizeof(float));
    float* weight = (float*) rd("course/ch03_build/weight.bin", (size_t)C*sizeof(float));
    float* mean   = (float*) rd("course/ch03_build/mean.bin",   (size_t)B*T*sizeof(float));
    float* rstd   = (float*) rd("course/ch03_build/rstd.bin",   (size_t)B*T*sizeof(float));
    float* dout   = (float*) rd("course/ch03_build/dout.bin",   (size_t)B*T*C*sizeof(float));
    float* dinp   = (float*) calloc((size_t)B*T*C, sizeof(float));
    layernorm_backward_dinp(dinp, dout, inp, weight, mean, rstd, B, T, C);
    FILE* f = fopen("course/ch03_build/dinp_ex3.bin","wb"); fwrite(dinp,4,(size_t)B*T*C,f); fclose(f);
    free(inp); free(weight); free(mean); free(rstd); free(dout); free(dinp); return 0;
}


In [ ]:
# Auto-grader for Exercise 3
import numpy as np, torch, torch.nn as nn, subprocess
torch.manual_seed(0); B, T, C = 2, 4, 8
x  = torch.randn(B, T, C, requires_grad=True)
ln = nn.LayerNorm(C); nn.init.normal_(ln.weight); nn.init.normal_(ln.bias)
out = ln(x); dout = torch.randn_like(out); out.backward(dout)
subprocess.run(["gcc","-O2","-Wall","-o","course/ch03_build/exercise3","course/ch03_build/exercise3.c"], check=True)
subprocess.run(["./course/ch03_build/exercise3", str(B), str(T), str(C)], check=True)
dinp_ex = np.fromfile("course/ch03_build/dinp_ex3.bin", dtype=np.float32).reshape(B, T, C)
err = np.max(np.abs(dinp_ex - x.grad.numpy()))
print(f"dinp max diff: {err:.2e}")
print("PASS" if err < 1e-5 else "FAIL — re-derive dval from the boxed formula")


### Solution to Exercise 3

In [ ]:
%%writefile course/ch03_build/exercise3_sol.c
#include <stdio.h>
#include <stdlib.h>

void layernorm_backward_dinp(float* dinp,
                             float* dout, float* inp, float* weight,
                             float* mean, float* rstd,
                             int B, int T, int C) {
    for (int b = 0; b < B; b++) {
        for (int t = 0; t < T; t++) {
            float* dout_bt = dout + b*T*C + t*C;
            float* inp_bt  = inp  + b*T*C + t*C;
            float* dinp_bt = dinp + b*T*C + t*C;
            float mean_bt  = mean[b*T + t];
            float rstd_bt  = rstd[b*T + t];

            float dnorm_mean = 0.0f, dnorm_norm_mean = 0.0f;
            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                float dnorm_i  = weight[i] * dout_bt[i];
                dnorm_mean      += dnorm_i;
                dnorm_norm_mean += dnorm_i * norm_bti;
            }
            dnorm_mean      /= C;
            dnorm_norm_mean /= C;

            for (int i = 0; i < C; i++) {
                float norm_bti = (inp_bt[i] - mean_bt) * rstd_bt;
                float dnorm_i  = weight[i] * dout_bt[i];
                float dval = (dnorm_i - dnorm_mean - norm_bti * dnorm_norm_mean) * rstd_bt;
                dinp_bt[i] += dval;
            }
        }
    }
}

static void* rd(const char* p, size_t n){FILE*f=fopen(p,"rb");void*b=malloc(n);size_t r=fread(b,1,n,f);(void)r;fclose(f);return b;}

int main(int argc, char** argv) {
    int B=atoi(argv[1]), T=atoi(argv[2]), C=atoi(argv[3]);
    float* inp    = (float*) rd("course/ch03_build/inp.bin",    (size_t)B*T*C*sizeof(float));
    float* weight = (float*) rd("course/ch03_build/weight.bin", (size_t)C*sizeof(float));
    float* mean   = (float*) rd("course/ch03_build/mean.bin",   (size_t)B*T*sizeof(float));
    float* rstd   = (float*) rd("course/ch03_build/rstd.bin",   (size_t)B*T*sizeof(float));
    float* dout   = (float*) rd("course/ch03_build/dout.bin",   (size_t)B*T*C*sizeof(float));
    float* dinp   = (float*) calloc((size_t)B*T*C, sizeof(float));
    layernorm_backward_dinp(dinp, dout, inp, weight, mean, rstd, B, T, C);
    FILE* f = fopen("course/ch03_build/dinp_ex3.bin","wb"); fwrite(dinp,4,(size_t)B*T*C,f); fclose(f);
    free(inp); free(weight); free(mean); free(rstd); free(dout); free(dinp); return 0;
}


In [ ]:
!gcc -O2 -Wall -o course/ch03_build/exercise3_sol course/ch03_build/exercise3_sol.c && ./course/ch03_build/exercise3_sol 2 4 8 && echo ran


## Recap

You now know:

- LayerNorm forward = mean → variance → rstd → normalize-scale-shift, all per-row.
- The `mean` / `rstd` cache is **the** trick: it's `O(B*T)` extra memory in exchange for skipping recomputation of two reductions per row in the backward.
- The backward needs **two passes** because every `dinp[i]` depends on `dnorm_mean` and `dnorm_norm_mean`, which themselves depend on every `j ≠ i`.
- `dweight` and `dbias` are scatter-add accumulators (every `(b,t)` contributes), just like `dwte`. `dinp` is per-position, but uses `+=` because the same buffer is fed by the residual path.
- The `dinp` formula
  $\;\;\partial L/\partial x_i = r\big(d\hat x_i - \overline{d\hat x} - \hat x_i \cdot \overline{d\hat x \cdot \hat x}\big)\;\;$
  is one of those derivations everybody re-derives once and then trusts. You've now done it once.

### What's next

**Chapter 4 — The Linear Layer (matmul).** This is where `llm.c` finally meets a "big" operation: every Transformer block has multiple `(B*T, C) @ (C, OC)` matmuls that dominate runtime. We'll see the naive triple loop, the `#pragma omp parallel for collapse(2)` + manual loop unrolling that buys ~5–10× speedup on CPU, why matmul is parallelism-friendly in *both* directions, and our first taste of why this op deserves a dedicated library (cuBLAS) on GPU.
